# ID6004W Industrial Vision - Assignment 4
### Lectures 14-21: Calibration, PnP, Epipolar Geometry, and Structure from Motion

| | |
|---|---|
| **Released** | Nov 23rd 2026 |
| **Due** | Dec 7th 2026 (5.00 PM IST) |
| **Total** | 100 marks (Theory 60 + Programming 40; will be scaled down appropriately later) |

## Instructions

**Working in Colab.** Open this notebook in Google Colab, and first use *File → Save a copy in Drive*. Work only in your copy. Type your answers in the cells marked **Your answer**, just below each question (double-click a cell to edit it). You can type mathematics in LaTeX between `$...$` or `$$...$$`. Photos of hand-written work are **not** accepted in place of typed answers.

- **The writing must be your own**, for both the theory answers and the code.
- When a question asks for a number, show the steps that lead to it.
- Cells marked **Optional check** are not graded. You may use them to check a hand calculation, but your answer must be worked out by hand, with the steps shown.
- Part B is done in this notebook, and the notebook is also your report. The code cells make the figures, and your written answers go in the **Your answer** cells of Part B. Before you submit, run every cell from top to bottom (*Runtime → Restart and run all*), so that all outputs and figures are saved in the file.
- **Submission.** Make a **single `.zip`** named `<ROLLNO>_assignment4.zip`. It must contain (1) this notebook with all outputs, downloaded with *File → Download → Download .ipynb* and named `<ROLLNO>_assignment4.ipynb`, and (2) the photos you took for Part B (the checkerboard photos and the two scene photos). Upload it here: [SUBMISSION LINK](https://forms.gle/N8D8w8hjxCDsHYQr7). For questions, contact Rishabh Shukla (da26e011@smail.iitm.ac.in). The form link will be disabled after the deadline. Follow the naming convention strictly.
- **Late policy:** 10 marks will be deducted for each day the assignment is submitted late after the due date.

## Dataset Setup Instructions

1. First, open this notebook in Google Colab.
2. Go to **File → Save a copy in Drive** and work only in your own copy of the notebook.
3. After creating your copy, upload the provided `assignment4_data.zip` file using the **Files** section in the left sidebar of Google Colab (folder icon → Upload).
4. Make sure the ZIP file is uploaded to the Colab session before running the setup cell below. Do not extract the ZIP file manually.
5. Run the setup cell. It will automatically validate and extract the dataset into the `assignment4_data` directory.
6. Once the setup is complete, execute the remaining cells in order.

**Note:** Colab runtime storage is temporary. If your runtime is restarted or disconnected, you may need to upload the ZIP file again and rerun the setup cell.

## Student details

- **Name:**
- **Roll number:**

In [3]:
# Assignment 4: Dataset Setup

import zipfile
from pathlib import Path

BASE = Path("/content")
DATA = BASE / "assignment4_data"

if not DATA.is_dir():

    # Find uploaded ZIP file in Colab
    zip_files = list(BASE.glob("assignment4_data*.zip"))

    if not zip_files:
        raise FileNotFoundError(
            "Please upload assignment4_data.zip using Colab's Files section."
        )

    zip_path = next(
        (p for p in zip_files if zipfile.is_zipfile(p)), None
    )

    print("Found:", zip_path)
    print("Extracting dataset...")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(BASE)

# Verify extraction
if not DATA.is_dir():
    raise FileNotFoundError("Dataset extraction failed.")

print("\nDataset setup successful!")
print(DATA, "->", sorted(p.name for p in DATA.iterdir()))



Dataset setup successful!
/content/assignment4_data -> ['checkerboard', 'sample_output', 'wadham']


In [4]:
import glob
import os
import sys
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV", cv2.__version__)       # SIFT needs OpenCV >= 4.4; Colab's default is fine

OpenCV 5.0.0


---
# Part A - Theory  [60 marks]

---
## Question 1. Calibration from one photo, and lens distortion  [8 marks]

**(a) Finding $K$ from three vanishing points.** **[5]**

You take one photo of the corner of a building. The building has edges in three directions, and the three directions are at right angles to each other. In the photo, the edges of each direction meet at one vanishing point:

$$
\mathbf v_1=(-1280,\,-1360),\qquad \mathbf v_2=(-80,\,1040),\qquad \mathbf v_3=(1120,\,-160).
$$

The camera has zero skew and square pixels, so

$$
K=\begin{bmatrix}f&0&u_0\\0&f&v_0\\0&0&1\end{bmatrix}.
$$

We want to find the focal length $f$ and the principal point $\mathbf p=(u_0,v_0)$.

1. Compute $\omega=K^{-\top}K^{-1}$. If two directions are perpendicular, their vanishing points satisfy $\tilde{\mathbf v}_i^{\top}\,\omega\,\tilde{\mathbf v}_j=0$, where $\tilde{\mathbf v}=(x,y,1)^{\top}$ is the point in homogeneous form. Multiply this out, and show that it is the same as
$$(\mathbf v_i-\mathbf p)\cdot(\mathbf v_j-\mathbf p)=-f^2 .$$
2. Write the equation of step 1 for the pair $(1,2)$ and for the pair $(1,3)$, and subtract them. The $f^2$ cancels, and you get $(\mathbf v_1-\mathbf p)\cdot(\mathbf v_2-\mathbf v_3)=0$. This equation is linear in $u_0$ and $v_0$. In the same way, use the pairs $(2,1)$ and $(2,3)$ to get a second linear equation. Solve the two equations for $u_0$ and $v_0$. Then use step 1 to find $f$.

**Your answer 1(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [ ]:
# Optional check for 1(a).
v1 = np.array([-1280.0, -1360.0])
v2 = np.array([-80.0, 1040.0])
v3 = np.array([1120.0, -160.0])

# TODO (optional): write your two linear equations as A @ p = b, solve with np.linalg.solve,
# then f = np.sqrt(-(v1 - p) @ (v2 - p)).

**(b) Lens distortion.** **[3]**

A camera has $f=800$, principal point $(u_0,v_0)=(320,240)$, and radial distortion $k_1=-0.2$. A straight edge in the world would appear in a perfect pinhole camera at the three normalised points $(0.5,\,0)$, $(0.5,\,0.25)$ and $(0.5,\,0.5)$. These three points lie on the straight line $x=0.5$. The lens moves each normalised point $(x_n,y_n)$ to

$$
x_d=x_n\,(1+k_1r^2),\qquad y_d=y_n\,(1+k_1r^2),\qquad\text{where } r^2=x_n^2+y_n^2 .
$$

1. Compute the three distorted points. Then convert them to pixels with $K$: the pixel is $(f\,x_d+u_0,\;f\,y_d+v_0)$.
2. Do the three pixels still lie on one straight line? Is this barrel or pincushion distortion?
3. Explain why no $3\times3$ matrix (not $K$, and not any other) can undo this distortion.

**Your answer 1(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [ ]:
# Optional check for 1(b).
k1, f, u0, v0 = -0.2, 800.0, 320.0, 240.0
xn = np.array([[0.5, 0.0], [0.5, 0.25], [0.5, 0.5]])     # one normalised point per row

# TODO (optional): r^2, the distorted points, their pixels, and the distance of the middle pixel
# from the line through the other two.

---
## Question 2. Camera pose from a projection matrix  [8 marks]

**(a) From $P$ to $R$ and $\mathbf t$.** **[5]**

A camera has $K=\begin{bmatrix}500&0&320\\0&500&240\\0&0&1\end{bmatrix}$. It sees six or more 3D points whose positions we know, and the DLT gives

$$
P=\begin{bmatrix}-416&0&-1112&-3060\\288&-1000&-384&-1720\\1.2&0&-1.6&-8\end{bmatrix}.
$$

1. Compute $M=K^{-1}P$. It should equal $\mu\,[R\mid\mathbf t]$ for some unknown number $\mu$. To find $\mu$, take the determinant of the left $3\times3$ block of $M$ (hint: $\det(\mu R)=\mu^3\det R=\mu^3$). Then write down $R$ and $\mathbf t$. Check that $R^{\top}R=I$ and $\det R=1$.
2. Find the camera centre $C=-R^{\top}\mathbf t$.
3. Why does the DLT need at least six points?

**Your answer 2(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [5]:
# Optional check for 2(a).
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], float)
P = np.array([[-416, 0, -1112, -3060],
              [288, -1000, -384, -1720],
              [1.2, 0, -1.6, -8]])

# TODO (optional): M = np.linalg.solve(K, P), mu = np.cbrt(det of M[:, :3]), then R, t and C.

**(b) RANSAC, and finding the translation.** **[3]**

1. RANSAC needs $N=\log(1-p)/\log(1-w^s)$ rounds. Here $w$ is the fraction of correct matches (inliers), $s$ is the number of points in one random sample, and $p$ is the confidence. Take $p=0.99$. Compute $N$ for $s=4$ and for $s=6$, first with $w=0.6$ and then with $w=0.3$. Which sample size is better?
2. Now suppose we already know $R$, and only $\mathbf t$ is unknown. A 3D point $\mathbf X$ is seen at $\tilde{\mathbf x}$ (normalised, homogeneous). Show that the equation $\tilde{\mathbf x}\times(R\mathbf X+\mathbf t)=\mathbf 0$ is linear in $\mathbf t$. How many point matches do we need to find $\mathbf t$?

**Your answer 2(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [6]:
# Optional check for 2(b).
p = 0.99

# TODO (optional): N = log(1 - p) / log(1 - w**s) for s in (4, 6) and w in (0.6, 0.3); round up.

---
## Question 3. Epipolar geometry, $F$ and $E$  [12 marks]

Two cameras look at the same scene. Both have $K=\mathrm{diag}(500,500,1)$, so pixel coordinates are measured from the principal point. The second camera is turned and moved with respect to the first by

$$
R=\begin{bmatrix}0.96&0&-0.28\\0&1&0\\0.28&0&0.96\end{bmatrix},\qquad \mathbf t=(-2,\;0,\;0.5)^{\top}.
$$

Recall that $[\mathbf t]_\times=\begin{bmatrix}0&-t_3&t_2\\t_3&0&-t_1\\-t_2&t_1&0\end{bmatrix}$ is the matrix with $[\mathbf t]_\times\mathbf a=\mathbf t\times\mathbf a$.

**(a) Building $F$.** **[4]**

1. Compute $[\mathbf t]_\times$, then $E=[\mathbf t]_\times R$, then $F=K^{-\top}EK^{-1}$. To keep the numbers simple, write your answer as $250000\,F$. Check that $\det F=0$.
2. Find the two epipoles: $\mathbf e$ in image 1, with $F\mathbf e=\mathbf 0$, and $\mathbf e'$ in image 2, with $F^{\top}\mathbf e'=\mathbf 0$. Give both in pixels.

**Your answer 3(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(b) Epipolar lines.** **[3]**

The point $\mathbf x=(50,25)$ is in the left image.

1. Compute its epipolar line $\boldsymbol\ell'=F\tilde{\mathbf x}$ in the right image. Check that it passes through $\mathbf e'$.
2. Two points in the right image might be its match: $\mathbf x'_A=(-200,24)$ and $\mathbf x'_B=(-200,30)$. Find the distance of each from $\boldsymbol\ell'$. (The distance of a point $(x,y)$ from the line $(a,b,c)$ is $|ax+by+c|/\sqrt{a^2+b^2}$.) Which one is within $1$ px of the line?

**Your answer 3(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(c) Depth from disparity.** **[2]**

A rectified stereo pair has $f=700$ px and baseline $b=0.12$ m, so the depth is $Z=fb/d$, where $d$ is the disparity.

1. Find the depth for $d=42$, $21$ and $4.2$ px.
2. Suppose each disparity is wrong by $0.5$ px. About how much does each depth change? (Use $\Delta Z\approx\dfrac{Z^2}{fb}\,\Delta d$.) What do you notice?

**Your answer 3(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [7]:
# Optional check for 3(a) - 3(c).
K = np.diag([500.0, 500.0, 1.0])
R = np.array([[0.96, 0, -0.28],
              [0, 1, 0],
              [0.28, 0, 0.96]])
t = np.array([-2.0, 0.0, 0.5])
x = np.array([50.0, 25.0, 1.0])
xA, xB = np.array([-200.0, 24.0, 1.0]), np.array([-200.0, 30.0, 1.0])

# TODO (optional): [t]_x, E, F (print 250000 * F), the epipoles from np.linalg.svd, the line F @ x
# and the two point-line distances; then Z = f b / d and the depth changes for 3(c).

**(d) The singular values of $E$.** **[3]**

1. Show that $EE^{\top}=\lVert\mathbf t\rVert^2I-\mathbf t\mathbf t^{\top}$. (Use $RR^{\top}=I$.) Then find what $EE^{\top}$ does to $\mathbf t$, and to a vector perpendicular to $\mathbf t$. Use this to show that the singular values of $E$ are $\lVert\mathbf t\rVert$, $\lVert\mathbf t\rVert$ and $0$.
2. How many degrees of freedom do $E$ and $F$ have? Explain briefly.

**Your answer 3(d).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 4. Estimating $F$  [12 marks]

**(a) The eight-point algorithm.** **[4]**

1. The matched points in one image are $(100,200)$, $(300,200)$, $(300,400)$ and $(100,400)$. Find Hartley's normalising matrix $T$. It moves the centroid (the mean point) to the origin, and then scales the points so that their mean distance from the origin is $\sqrt2$. Also give the four normalised points.
2. For the match $\mathbf x=(300,400)\leftrightarrow\mathbf x'=(310,380)$, write its row of the matrix $A$,
$$[\,x'x,\;x'y,\;x',\;y'x,\;y'y,\;y',\;x,\;y,\;1\,],$$
first in pixels, and then after normalising (use the same $T$ for both images). Why is the normalised row better?
3. The null vector of $A$, reshaped to $3\times3$, is $\hat F=U\,\mathrm{diag}(4,\,2,\,0.5)\,V^{\top}$. Which matrix should we use as $F$? Why?

**Your answer 4(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [8]:
# Optional check for 4(a).
pts = np.array([[100, 200], [300, 200], [300, 400], [100, 400]], float)
x, xp = np.array([300.0, 400.0]), np.array([310.0, 380.0])

# TODO (optional): the centroid, the scale s, T, the normalised points, and the two rows of A.

**(b) Algebraic error and Sampson distance.** **[4]**

For a rectified pair, $F=\begin{bmatrix}0&0&0\\0&0&-1\\0&1&0\end{bmatrix}$ (pixel coordinates measured from the principal point). One match is $\mathbf x=(100,50)\leftrightarrow\mathbf x'=(60,53)$.

1. Compute the algebraic error $\tilde{\mathbf x}'^{\top}F\tilde{\mathbf x}$. Then find the distance of $\mathbf x'$ from its epipolar line $F\tilde{\mathbf x}$.
2. Compute the Sampson distance $d$, where
$$d^2=\frac{(\tilde{\mathbf x}'^{\top}F\tilde{\mathbf x})^2}{(F\tilde{\mathbf x})_1^2+(F\tilde{\mathbf x})_2^2+(F^{\top}\tilde{\mathbf x}')_1^2+(F^{\top}\tilde{\mathbf x}')_2^2}.$$
Why is it smaller than the distance in step 1?

**Your answer 4(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [9]:
# Optional check for 4(b).
F = np.array([[0, 0, 0], [0, 0, -1], [0, 1, 0]], float)
x, xp = np.array([100.0, 50.0, 1.0]), np.array([60.0, 53.0, 1.0])

# TODO (optional): the algebraic error xp @ F @ x, the line F @ x, and the Sampson distance.

**(c) When $F$ cannot be found.** **[4]**

1. A student photographs a poster on a wall twice, from two places. All the matched points lie on one plane, so $\tilde{\mathbf x}'\sim H\tilde{\mathbf x}$ for a homography $H$. Show that $F=[\mathbf v]_\times H$ satisfies $\tilde{\mathbf x}'^{\top}F\tilde{\mathbf x}=0$ for every match and for **every** vector $\mathbf v$. (Hint: $\mathbf v\times\tilde{\mathbf x}'$ is perpendicular to $\tilde{\mathbf x}'$.) What does this mean for estimating $F$?
2. Another student only turns the camera between the two photos, without moving it ($\mathbf t=\mathbf 0$). What is $E$? Can we estimate $F$? Can we triangulate points?

**Your answer 4(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 5. Two-view reconstruction  [12 marks]

**(a) Four poses from $E$.** **[5]**

An essential matrix and its SVD $E=U\Sigma V^{\top}$ are

$$
E=\begin{bmatrix}0&0&0\\-0.6&0&0.8\\0&-1&0\end{bmatrix},\qquad
U=\begin{bmatrix}0&0&-1\\1&0&0\\0&1&0\end{bmatrix},\qquad
V^{\top}=\begin{bmatrix}-0.6&0&0.8\\0&-1&0\\0.8&0&0.6\end{bmatrix},
$$

with $\Sigma=\mathrm{diag}(1,1,0)$. Let $W=\begin{bmatrix}0&-1&0\\1&0&0\\0&0&1\end{bmatrix}$.

1. Compute $\det U$. Then compute $UWV^{\top}$ and its determinant. Why is it not a rotation? Show that changing the sign of the third column of $U$ fixes this, and does not change $E$.
2. With the fixed $U$, write down the two rotations $R_a=UWV^{\top}$ and $R_b=UW^{\top}V^{\top}$, and the two translations $\mathbf t=\pm\mathbf u_3$, where $\mathbf u_3$ is the third column of $U$. Together they give four candidate poses.
3. A point is seen at the normalised coordinates $\hat{\mathbf x}=(0,\,0.2)$ in view 1 and $\hat{\mathbf x}'=(0.5,\,0.25)$ in view 2. For each of the four candidates, solve $z'\,\hat{\mathbf x}'=z\,R\,\hat{\mathbf x}+\mathbf t$ for the two depths $z$ and $z'$ (write $\hat{\mathbf x}$ and $\hat{\mathbf x}'$ as 3-vectors with last entry $1$). Which candidate puts the point in front of both cameras? Give the 3D point.

**Your answer 5(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [10]:
# Optional check for 5(a).
E = np.array([[0, 0, 0], [-0.6, 0, 0.8], [0, -1, 0]], float)
U = np.array([[0, 0, -1], [1, 0, 0], [0, 1, 0]], float)
Vt = np.array([[-0.6, 0, 0.8], [0, -1, 0], [0.8, 0, 0.6]])
W = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]], float)
x1, x2 = np.array([0, 0.2, 1]), np.array([0.5, 0.25, 1])

# TODO (optional): fix the sign of U, form Ra, Rb and the four (R, t); for each, solve
# [x2, -R @ x1] @ [z', z] = t with np.linalg.lstsq and look at the signs of z and z'.

**(b) Triangulation.** **[3]**

Take the cameras $P=[\,I\mid\mathbf 0\,]$ and $P'=[\,I\mid(-1,0,0)^{\top}]$, and the match $\hat{\mathbf x}=(0.25,\,0.1)\leftrightarrow\hat{\mathbf x}'=(0.05,\,0.1)$. Each view gives two rows, $\hat x\,\mathbf p_3^{\top}-\mathbf p_1^{\top}$ and $\hat y\,\mathbf p_3^{\top}-\mathbf p_2^{\top}$, where $\mathbf p_k^{\top}$ is row $k$ of that camera matrix.

1. Write down the $4\times4$ matrix $A$, and show that it has rank $3$.
2. Solve $A\tilde{\mathbf X}=\mathbf 0$ for the 3D point $\mathbf X$. Check that its depth equals baseline divided by disparity.

**Your answer 5(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [ ]:
# Optional check for 5(b).
P1 = np.hstack([np.eye(3), np.zeros((3, 1))])
P2 = np.hstack([np.eye(3), [[-1], [0], [0]]])
x1, x2 = np.array([0.25, 0.1]), np.array([0.05, 0.1])

# TODO (optional): stack the four rows into A, check np.linalg.matrix_rank(A), and take the
# last right singular vector of A.

**(c) Scale.** **[2]**

Show that if every 3D point $\mathbf X_j$ is replaced by $s\mathbf X_j$, and $\mathbf t$ by $s\mathbf t$, then every image point stays the same. What extra information would we need to find $s$?

**Your answer 5(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(d) What can be recovered.** **[2]**

When $K$ is known, a two-view reconstruction is correct only up to some transformation of 3D space. Which transformation is it, and how many degrees of freedom does it have? What if $K$ is not known?

**Your answer 5(d).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
## Question 6. Many views: adding cameras, and bundle adjustment  [8 marks]

**(a) Adding a third view.** **[3]**

A student reconstructs views 1 and 2 first, and sets the distance between their two camera centres to $1$. In this reconstruction, a point seen in views 1, 2 and 3 has depth $6.0$ in camera 2. The student then tries another way: they compute $E$ from views 2 and 3 alone, set $\lVert\mathbf t_{23}\rVert=1$, and find the same point at depth $4.0$ in camera 2.

1. In the units of the first reconstruction, what is the distance between camera centres 2 and 3?
2. Why is it better to add view 3 with PnP, using the 3D points that are already reconstructed?

**Your answer 6(a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(b) The size of bundle adjustment.** **[3]**

A reconstruction with known $K$ has $m=20$ cameras and $n=5000$ points. On average, each point is seen in $4$ views.

1. How many unknowns are there? (Each camera has $6$, each point has $3$.) How many scalar residuals are there? (Each observation gives $2$.) What fraction of all (camera, point) pairs are actually observed?
2. If we move, turn or scale the whole scene and all the cameras together, the cost does not change. How many degrees of freedom is this? How are they removed in practice?

**Your answer 6(b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**(c) A robust loss.** **[2]**

One wrong match has a residual of $r=50$ px. Compute its cost $\rho(r)$ and its pull $\rho'(r)$ on the solution for two losses:

- the squared loss $\rho(r)=r^2$;
- the Huber loss with $\delta=2$ px: $\rho(r)=r^2$ if $|r|\le\delta$, and $\rho(r)=2\delta|r|-\delta^2$ otherwise.

Which loss is better here, and why?

**Your answer 6(c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

In [11]:
# Optional check for 6(b) and 6(c).
m, n, views_per_point = 20, 5000, 4
r, delta = 50.0, 2.0

# TODO (optional): unknowns, residuals and the observed fraction; the squared and Huber costs and pulls.

---
# Part B - Programming  [40 marks]

## Question 7. Two-view reconstruction  [40 marks]

You will calibrate your phone camera, and then build a 3D point cloud from two photos of a scene. Most of the code is given. You write seven short functions.

**The steps.**
1. **Calibrate** (part a, given): photos of a checkerboard give $K$ and the lens distortion.
2. **Match** (given): SIFT finds the same points in both photos.
3. **$F$** (part b): the eight-point algorithm inside RANSAC finds $F$ and throws away the wrong matches.
4. **Pose** (part c): $E=K_2^{\top}FK_1$ gives four possible poses, and chirality picks the right one.
5. **Points** (part d): triangulation turns each match into a 3D point.

**Your photographs.** Use your phone's main camera. Hold the phone in landscape (sideways) for every photo, and do not zoom.
- $12$-$15$ photos of a checkerboard (printed on paper, or shown on a laptop screen). Tilt the board differently in each photo.
- $2$ photos of a scene that is **not** flat (for example the corner of a building, or a table with objects on it). Between the two photos, take a small step sideways.
- Save them as JPEG. (On an iPhone, set *Settings → Camera → Formats → Most Compatible*; HEIC files cannot be read.) Put all the photos in the zip.

**Sample data.** The setup cell downloaded `assignment4_data/`. Get your code working on it first.
- `wadham/001.jpg` … `005.jpg`: five photos of Wadham College, Oxford, taken by a person walking along the courtyard. The scene has two walls at right angles, so it is not flat. `K1.txt` … `K5.txt` hold the intrinsic matrix of each photo, and `001.P` … `005.P` the true camera matrices, which are used **only** to print how far your pose is from the truth. Source: the Oxford Visual Geometry Group multi-view data (Werner and Zisserman, ECCV 2002); see `wadham/README_VGG.txt`.
- `checkerboard/left01.jpg` … `left14.jpg`: thirteen photos of a board with $9\times6$ inner corners, from OpenCV's sample data.
- `sample_output/`: what the finished code writes for these photos. RANSAC is seeded, so if your functions are right, your numbers should match the `report.txt` files, often to the last digit. Small differences are fine.

**Starter code.** The cells below hold the starter code, split by part. Fill in the seven functions marked `TODO`: `epipolar_rows`, `enforce_rank2`, `eight_point`, `sampson_distance`, `decompose_E`, `choose_pose` and `triangulate`. Do not change anything else. Use NumPy only for these seven functions, and work on all the matches at once (no Python loop over the matches).

**You may not use** `cv2.findFundamentalMat`, `cv2.findEssentialMat`, `cv2.recoverPose`, `cv2.decomposeEssentialMat`, `cv2.triangulatePoints` or `cv2.findHomography` in your functions. The given code calls some of them, but **only** to print a comparison with your results.

In [13]:
RNG = np.random.default_rng(0)

### Helpers (given, do not change)

In [12]:
def load_images(paths, max_side):
    """Read, and downscale so the longer side is at most max_side. Returns the
    images and the scale factor applied to each."""
    ims, scales = [], []
    for p in paths:
        im = cv2.imread(p, cv2.IMREAD_COLOR)
        if im is None:
            raise FileNotFoundError(f"cannot read {p}")
        s = max_side / max(im.shape[:2])
        if s < 1:
            im = cv2.resize(im, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        else:
            s = 1.0
        ims.append(im)
        scales.append(s)
    return ims, scales


def to_h(p):
    """(n,2) -> (n,3) homogeneous."""
    return np.hstack([p, np.ones((len(p), 1))])


def from_h(ph):
    return ph[:, :-1] / ph[:, -1:]


def skew(v):
    return np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]], float)


def rotation_angle_deg(R):
    return np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))


def angle_between_deg(a, b):
    return np.degrees(np.arccos(np.clip(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1)))


def adaptive_iterations(w, s, p=0.99, lo=50, hi=20000):
    """N = log(1-p) / log(1-w^s), guarded and clipped to [lo, hi]."""
    ws = w ** s
    if ws <= 1e-12:
        return hi
    if ws >= 1 - 1e-12:
        return lo
    return int(np.clip(np.ceil(np.log(1 - p) / np.log(1 - ws)), lo, hi))


def project(K, R, t, X):
    return from_h((X @ R.T + t) @ K.T)


def depth_in(R, t, X):
    """Z coordinate of every X in the camera (R, t)."""
    return X @ R[2] + t[2]


def centre(R, t):
    return -R.T @ t


def side_by_side(im1, im2):
    h = max(im1.shape[0], im2.shape[0])
    out = np.zeros((h, im1.shape[1] + im2.shape[1], 3), np.uint8)
    out[:im1.shape[0], :im1.shape[1]] = im1
    out[:im2.shape[0], im1.shape[1]:] = im2
    return out

In [14]:
def show_image(path, title=None, width=16):
    """Display an image file inline (BGR on disk -> RGB for matplotlib)."""
    im = cv2.imread(path)
    if im is None:
        print("missing:", path)
        return
    h, w = im.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(title or os.path.basename(path))
    plt.axis("off")
    plt.show()


def upload_to(folder):
    # Upload files from your computer into `folder` (Colab only); returns the sorted list of image paths.
    os.makedirs(folder, exist_ok=True)
    try:
        from google.colab import files
        for name, data in files.upload().items():
            with open(os.path.join(folder, name), "wb") as f:
                f.write(data)
    except ImportError:
        print(f"not running in Colab: copy the images into {folder}/ by hand")
    paths = sorted(p for p in glob.glob(os.path.join(folder, "*"))
                   if p.lower().endswith((".jpg", ".jpeg", ".png")))
    print(paths)
    return paths

### (a) Calibration  [4 marks]

There is no code to write in this part. `run_calib` finds the inner corners of the checkerboard in every photo, and calls OpenCV to estimate $K$ and the lens distortion $(k_1,k_2,p_1,p_2,k_3)$. It saves them to `K.txt` and `dist.txt`, which the later parts read. Run it on the sample board first, and then on your own board.

In [15]:
def calibrate(paths, pattern, square, max_side, out):
    ims, _ = load_images(paths, max_side)
    h, w = ims[0].shape[:2]
    if any(im.shape[:2] != (h, w) for im in ims):
        raise ValueError("all calibration images must have the same size and orientation")
    cols, rows = pattern
    obj = np.zeros((rows * cols, 3), np.float32)
    obj[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2) * square
    obj_pts, img_pts, used = [], [], []
    crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 1e-4)
    for p, im in zip(paths, ims):
        g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        ok, c = cv2.findChessboardCorners(g, (cols, rows))
        if not ok:
            print(f"  no board found in {os.path.basename(p)}, skipped")
            continue
        c = cv2.cornerSubPix(g, c, (5, 5), (-1, -1), crit)
        obj_pts.append(obj)
        img_pts.append(c)
        used.append(p)
    if len(used) < 3:
        raise ValueError("need the board in at least three views, at different tilts")
    rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(obj_pts, img_pts, (w, h), None, None)
    per_view = []
    for o, ip, rv, tv in zip(obj_pts, img_pts, rvecs, tvecs):
        proj, _ = cv2.projectPoints(o, rv, tv, K, dist)
        per_view.append(np.sqrt(((proj - ip) ** 2).sum(-1).mean()))
    return dict(K=K, dist=dist.ravel(), rms=rms, per_view=per_view, used=used, size=(w, h),
                ims=ims, img_pts=img_pts)


def save_K(path, K, size):
    np.savetxt(path, K, fmt="%.6f", header=f"width {size[0]} height {size[1]}")


def load_K(path, width, height):
    """Read K and rescale it if the header states a different image width. Refuse
    a K whose image had a different aspect ratio (e.g. portrait against landscape)."""
    K = np.loadtxt(path)
    w0 = h0 = None
    with open(path) as f:
        for line in f:
            if line.startswith("#") and "width" in line:
                toks = line.replace("#", " ").split()
                w0 = float(toks[toks.index("width") + 1])
                if "height" in toks:
                    h0 = float(toks[toks.index("height") + 1])
    if h0 is not None and abs(w0 / h0 - width / height) > 0.01:
        raise ValueError(f"{path} is for {w0:.0f}x{h0:.0f} images but these are {width}x{height}: "
                 "calibrate with the phone held the same way as for the scene")
    if w0 is not None and abs(w0 - width) > 0.5:
        K = K.copy()
        K[:2] *= width / w0
    return K, w0


def run_calib(paths, pattern, square=1.0, out="a4_calib", max_side=1000, show=True):
    """Given, do not change. Calibrates from the checkerboard photographs in `paths`. `pattern` is the number
    of INNER corners of the board, columns x rows, e.g. "9x6". Writes out/K.txt and out/dist.txt (for the
    images at the downscaled size), the corner figure and out/report.txt, and shows the figure."""
    os.makedirs(out, exist_ok=True)
    lines = []

    def say(s=""):
        print(s)
        lines.append(s)

    cols, rows = map(int, pattern.lower().split("x"))
    r = calibrate(paths, (cols, rows), square, max_side, out)
    K, dist = r["K"], r["dist"]
    w, h = r["size"]
    say(f"(a) {len(r['used'])}/{len(paths)} views with the board found, images {w}x{h}")
    say("(a) K =\n" + np.array2string(K, precision=2, suppress_small=True))
    say(f"(a) distortion (k1, k2, p1, p2, k3) = {np.array2string(dist, precision=4)}")
    say(f"(a) RMS reprojection error {r['rms']:.3f} px; per view: "
        + ", ".join(f"{e:.2f}" for e in r["per_view"]))
    fov = 2 * np.degrees(np.arctan(w / (2 * K[0, 0])))
    say(f"(a) horizontal field of view {fov:.1f} deg, aspect fy/fx = {K[1, 1] / K[0, 0]:.4f}, "
        f"principal point offset from centre ({K[0, 2] - w / 2:.1f}, {K[1, 2] - h / 2:.1f}) px")
    save_K(os.path.join(out, "K.txt"), K, (w, h))
    np.savetxt(os.path.join(out, "dist.txt"), dist[None], fmt="%.8f")
    im = r["ims"][0]
    vis = cv2.drawChessboardCorners(im.copy(), (cols, rows), r["img_pts"][0], True)
    und = cv2.undistort(im, K, dist)
    cv2.imwrite(os.path.join(out, "a_corners_and_undistorted.jpg"), side_by_side(vis, und))
    say(f"wrote {out}/K.txt and {out}/dist.txt")
    with open(os.path.join(out, "report.txt"), "w") as f:
        f.write("\n".join(lines) + "\n")
    if show:
        show_image(os.path.join(out, "a_corners_and_undistorted.jpg"),
                   "left: the corners found in the first photo; right: the same photo with the distortion removed")
    return r

**The sample board.** Run the calibration on the 13 sample photos, and compare with the reference `report.txt` printed below it.

In [ ]:
board = sorted(glob.glob(f"{DATA}/checkerboard/*.jpg"))
calib_sample = run_calib(board, pattern="9x6", out="out_calib")

In [ ]:
print(open(f"{DATA}/sample_output/calib/report.txt").read())

**Your board.** Upload your $12$-$15$ checkerboard photos with the next cell (or drag them into a folder `my_checkerboard/` with the file browser on the left). Then set `PATTERN` to the number of **inner** corners of your board, columns × rows. For example, a board with $10\times7$ squares has $9\times6$ inner corners.

In [ ]:
my_board = upload_to("my_checkerboard")

In [ ]:
PATTERN = "9x6"           # inner corners of YOUR board, columns x rows
assert len(my_board) >= 3, "upload your checkerboard photos with the cell above first"
calib_mine = run_calib(my_board, pattern=PATTERN, out="a4_calib")

**Report (a).** Answer from the run on **your own board**. Give $K$ and the RMS reprojection error, and refer to the corner figure above. Why must lens distortion be removed before the next steps? (See Question 1(b).)

**Your answer (a).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

### Features and matches (given, do not change)

SIFT finds up to 8000 keypoints in each photo. A match is kept only if it passes Lowe's ratio test at $0.8$ **and** the two keypoints are each other's nearest neighbour. `keypoint_xy` returns the pixel coordinates of the matched keypoints, with the lens distortion removed.

In [21]:
def sift_features(im):
    sift = cv2.SIFT_create(nfeatures=8000)
    kp, des = sift.detectAndCompute(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), None)
    return kp, des


def match_pair(des1, des2, ratio=0.8):
    """Lowe's ratio test, plus a mutual nearest-neighbour check. Returns a list of
    (index in image 1, index in image 2)."""
    bf = cv2.BFMatcher(cv2.NORM_L2)
    fwd = bf.knnMatch(des1, des2, k=2)
    bwd = bf.match(des2, des1)
    back = {m.queryIdx: m.trainIdx for m in bwd}
    good = [(m.queryIdx, m.trainIdx) for m, n in (p for p in fwd if len(p) == 2)
            if m.distance < ratio * n.distance and back.get(m.trainIdx) == m.queryIdx]
    return good


def keypoint_xy(kp, idx, K, dist):
    """Pixel coordinates of keypoints idx, with lens distortion removed if dist is
    given (cv2.undistortPoints with P=K returns ideal pinhole pixels)."""
    p = np.float64([kp[i].pt for i in idx])
    if dist is not None and np.any(dist != 0):
        p = cv2.undistortPoints(p.reshape(-1, 1, 2), K, dist, P=K).reshape(-1, 2)
    return p

### (b) The fundamental matrix  [14 marks]

Write these four functions. The comments in the code say exactly what each one must return.

1. `epipolar_rows(x1, x2)`: one row $[\,x'x,\;x'y,\;x',\;y'x,\;y'y,\;y',\;x,\;y,\;1\,]$ for every match, where $(x,y)$ is the point in image 1 and $(x',y')$ is its match in image 2. The result has shape $(m,9)$. Do not swap $\mathbf x$ and $\mathbf x'$.
2. `enforce_rank2(F)`: take the SVD of $F$, set the smallest singular value to $0$, and multiply back.
3. `eight_point(x1, x2)`: the normalised eight-point algorithm. The normalisation (`normalise_points`) is given. Build $A$ from the normalised points, take the last right singular vector of $A$, reshape it to $3\times3$ row by row, make it rank 2, undo the normalisation with $F=T_2^{\top}\tilde F\,T_1$, and scale it so that $\lVert F\rVert=1$.
4. `sampson_distance(F, x1, x2)`: the Sampson distance of every match, in pixels. This is $d$ from Question 4(b), not $d^2$.

The RANSAC loop `ransac_F` and the epipoles are given. RANSAC draws eight matches at a time, counts a match as an inlier when its Sampson distance is below $1$ px, and at the end fits $F$ again on all the inliers.

**To do:** `epipolar_rows`, `enforce_rank2`, `eight_point`, `sampson_distance`.

In [ ]:
def normalise_points(p):
    """Hartley: centroid to the origin, mean distance sqrt(2). Returns (n,2) points and T."""
    c = p.mean(axis=0)
    d = np.sqrt(((p - c) ** 2).sum(axis=1)).mean()
    s = np.sqrt(2) / max(d, 1e-12)
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    return from_h(to_h(p) @ T.T), T


def epipolar_rows(x1, x2):
    """One row [x'x, x'y, x', y'x, y'y, y', x, y, 1] per correspondence x1 -> x2."""
    # TODO (b): one row of the 8-point system per correspondence, shape (m, 9)
    return None


def enforce_rank2(F):
    """Nearest rank-2 matrix in the Frobenius norm (Eckart-Young): zero sigma3."""
    # TODO (b)
    return None


def eight_point(x1, x2, normalise=True):
    """F with x2^T F x1 = 0 from m >= 8 correspondences: normalise, null vector of
    A, rank-2 correction, denormalise F = T2^T F~ T1. Returned with ||F||_F = 1."""
    if normalise:
        n1, T1 = normalise_points(x1)
        n2, T2 = normalise_points(x2)
    else:
        n1, n2, T1, T2 = x1, x2, np.eye(3), np.eye(3)
    # TODO (b): build A from (n1, n2), take the last right singular vector,
    # reshape it row by row, enforce rank 2, undo the normalisation, scale to ||F|| = 1
    F = None
    return F


def eight_point_singular_values(x1, x2, normalise=True):
    if normalise:
        x1, _ = normalise_points(x1)
        x2, _ = normalise_points(x2)
    return np.linalg.svd(epipolar_rows(x1, x2), compute_uv=False)


def sampson_distance(F, x1, x2):
    """Sampson distance in pixels (not squared) for every correspondence."""
    # TODO (b): with Fx1 = F x1 and Ftx2 = F^T x2 for every correspondence,
    #   d^2 = (x2^T F x1)^2 / ((Fx1)_1^2 + (Fx1)_2^2 + (Ftx2)_1^2 + (Ftx2)_2^2)
    # vectorised over all m correspondences; return d, shape (m,)
    return None

In [ ]:
def ransac_F(x1, x2, tol=1.0, p=0.99, max_iter=20000, min_iter=100):
    """Eight-point samples, consensus by Sampson distance < tol pixels, adaptive N
    with s = 8, then the normalised eight-point algorithm on all inliers and a
    final consensus of the refit.
    Returns F, a boolean inlier mask, and the number of iterations run."""
    m = len(x1)
    if m < 8:
        raise ValueError("need at least eight correspondences")
    best_inl = np.zeros(m, bool)
    N, it = max_iter, 0
    while it < N:
        it += 1
        idx = RNG.choice(m, 8, replace=False)
        F = eight_point(x1[idx], x2[idx])
        inl = sampson_distance(F, x1, x2) < tol
        if inl.sum() > best_inl.sum():
            best_inl = inl
            N = adaptive_iterations(inl.mean(), 8, p, min_iter, max_iter)
    F = eight_point(x1[best_inl], x2[best_inl])
    inl = sampson_distance(F, x1, x2) < tol
    F = eight_point(x1[inl], x2[inl])
    inl = sampson_distance(F, x1, x2) < tol
    return F, inl, it


def epipoles(F):
    """Right null vector e (F e = 0) and left null vector e' (F^T e' = 0)."""
    _, _, Vt = np.linalg.svd(F)
    e = Vt[-1]
    _, _, Vt2 = np.linalg.svd(F.T)
    ep = Vt2[-1]
    return e, ep

In [ ]:
def synthetic_scene(rng):
    """Given. A known scene: K, the pose (R, t) of camera 2, 40 points X, their exact images x1 and x2 (no
    noise), and the true F scaled to ||F|| = 1."""
    K = np.array([[800, 0, 320], [0, 780, 240], [0, 0, 1.0]])
    ang = np.radians(12)
    R = np.array([[np.cos(ang), 0, np.sin(ang)], [0, 1, 0], [-np.sin(ang), 0, np.cos(ang)]])
    t = np.array([-1.0, 0.1, 0.2])
    X = rng.uniform([-2, -1.5, 4], [2, 1.5, 9], (40, 3))
    x1 = project(K, np.eye(3), np.zeros(3), X)
    x2 = project(K, R, t, X)
    F_true = np.linalg.inv(K).T @ skew(t) @ R @ np.linalg.inv(K)
    F_true /= np.linalg.norm(F_true)
    return K, R, t, X, x1, x2, F_true


def sign_free_diff(A, B):
    """Largest entry-wise difference between A and B, allowing for a change of sign."""
    return min(np.abs(A - B).max(), np.abs(A + B).max())


def synthetic_F_test():
    """Given. Your eight-point F on exact data, from all 40 points and from the first eight."""
    K, R, t, X, x1, x2, F_true = synthetic_scene(np.random.default_rng(0))
    return sign_free_diff(eight_point(x1, x2), F_true), sign_free_diff(eight_point(x1[:8], x2[:8]), F_true)


def synthetic_tests():
    """Exact data from a known scene: the eight-point F (from all the points and
    from a minimal sample of eight), E's decomposition with chirality, and
    triangulation must all recover the truth to ~1e-8 or better."""
    K, R, t, X, x1, x2, F_true = synthetic_scene(RNG)
    sgn = sign_free_diff
    F8 = eight_point(x1, x2)
    F8min = eight_point(x1[:8], x2[:8])
    Kn = np.linalg.inv(K)
    x1n, x2n = from_h(to_h(x1) @ Kn.T), from_h(to_h(x2) @ Kn.T)
    Rc, tc, counts = choose_pose(essential_from_F(F8, K, K), x1n, x2n)
    eR = np.abs(Rc - R).max()
    et = np.abs(tc - t / np.linalg.norm(t)).max()
    P1 = np.hstack([np.eye(3), np.zeros((3, 1))])
    Xt = triangulate(P1, np.hstack([R, t[:, None]]), x1n, x2n)
    return dict(F8=sgn(F8, F_true), F8min=sgn(F8min, F_true), R=eR, t=et, counts=counts,
                X=np.abs(Xt - X).max())

In [ ]:
# Check (b) before you use real photographs. The matches here are exact (no noise), so your eight_point
# must give back the true F. Both numbers should be about 1e-12 or smaller.
e_all, e_eight = synthetic_F_test()
print(f"eight-point on exact data: |F - F_true| = {e_all:.1e} with all 40 points, {e_eight:.1e} with only eight")

### (c) The essential matrix and the pose  [9 marks]

`essential_from_F` is given: it computes $E=K_2^{\top}FK_1$ and sets its singular values to $(s,s,0)$. Write:

1. `decompose_E(E)`: the four candidate poses of Question 5(a). Take the SVD of $E$. If $\det U<0$, change the sign of the third column of $U$. If $\det V^{\top}<0$, change the sign of the third row of $V^{\top}$. Return the list `[(Ra, u3), (Ra, -u3), (Rb, u3), (Rb, -u3)]` in this order, with $R_a=UWV^{\top}$, $R_b=UW^{\top}V^{\top}$, and $\mathbf u_3$ the third column of $U$.
2. `choose_pose(E, x1n, x2n)`: for each candidate, triangulate all the matches with $P_1=[I\mid\mathbf 0]$ and $P_2=[R\mid\mathbf t]$, and count the points with positive depth in **both** cameras (`depth_in` is given). Return the candidate with the largest count, and the list of the four counts.

`choose_pose` calls your `triangulate` from part (d), so write both before you run the check below part (d).

**To do:** `decompose_E`, `choose_pose`.

In [ ]:
def essential_from_F(F, K1, K2):
    """E = K2^T F K1 with the singular values replaced by ((s1+s2)/2, (s1+s2)/2, 0)."""
    E = K2.T @ F @ K1
    U, S, Vt = np.linalg.svd(E)
    s = (S[0] + S[1]) / 2
    return U @ np.diag([s, s, 0.0]) @ Vt


def decompose_E(E):
    """The four candidates (R, t), R in {U W V^T, U W^T V^T}, t = +-u3, with the
    signs of U and V^T fixed so that both rotations have det = +1. ||t|| = 1."""
    # TODO (c): take the SVD of E; if det(U) < 0 negate the third column of U,
    # if det(V^T) < 0 negate the third row of V^T (harmless, since sigma3 = 0)
    W = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1.0]])
    return []


def choose_pose(E, x1n, x2n):
    """Triangulate with each of the four candidates (in normalised coordinates,
    P1 = [I|0], P2 = [R|t]) and keep the one that puts the most points in front
    of both cameras. Returns (R, t, counts)."""
    P1 = np.hstack([np.eye(3), np.zeros((3, 1))])
    # TODO (c): for each (R, t) in decompose_E(E): triangulate, count the points
    # with depth_in(...) > 0 in camera 1 AND in camera 2, keep the best.
    # counts must be in the order decompose_E returns the candidates.
    counts = []
    R, t = None, None
    return R, t, counts

### (d) Triangulation  [7 marks]

Write `triangulate(P1, P2, x1, x2)`, the linear (DLT) triangulation of Question 5(b), for all the matches at once. For each match, stack the two rows $x\,\mathbf p_3^{\top}-\mathbf p_1^{\top}$ and $y\,\mathbf p_3^{\top}-\mathbf p_2^{\top}$ of camera 1 and the same two rows of camera 2 into a $4\times4$ matrix $A$. Take the last right singular vector of $A$, and divide it by its last entry. `np.linalg.svd` works on a stack of matrices of shape $(n,4,4)$, so no loop is needed.

**To do:** `triangulate`.

In [ ]:
def triangulate(P1, P2, x1, x2):
    """Linear (DLT) triangulation of every correspondence. For each point stack
    x p3^T - p1^T, y p3^T - p2^T for both cameras into a 4x4 A and take its last
    right singular vector. Returns (n,3) inhomogeneous points. Vectorised."""
    # TODO (d): build A with shape (n,4,4); np.linalg.svd works on a stack of
    # matrices. Dividing each row of A by its norm before the SVD is a cheap
    # improvement in conditioning and is recommended.
    return None

In [ ]:
# Check (b) - (d) together on exact data. One chirality count should be 40 and the other three 0,
# and every error should be about 1e-12 or smaller.
s = synthetic_tests()
print(f"chirality counts {s['counts']}, |R - R_true| = {s['R']:.1e}, |t - t_true| = {s['t']:.1e}, "
      f"triangulation |X - X_true| = {s['X']:.1e}")

### Figures and the driver (given, do not change)

In [ ]:
def draw_inliers(im1, im2, x1, x2, inl, max_lines=400):
    out = side_by_side(im1, im2)
    off = im1.shape[1]
    idx = np.arange(len(x1))
    if len(idx) > max_lines:
        idx = np.sort(RNG.choice(len(x1), max_lines, replace=False))
    for i in idx:
        col = (0, 200, 0) if inl[i] else (0, 0, 255)
        a = tuple(np.round(x1[i]).astype(int))
        b = (int(round(x2[i, 0] + off)), int(round(x2[i, 1])))
        cv2.line(out, a, b, col, 1, cv2.LINE_AA)
        cv2.circle(out, a, 3, col, -1)
        cv2.circle(out, b, 3, col, -1)
    return out


def line_endpoints(l, w, h):
    """Clip the line l = (a,b,c) to the image rectangle; None if it misses."""
    a, b, c = l
    pts = []
    if abs(b) > 1e-12:
        for x in (0, w - 1):
            y = -(a * x + c) / b
            if 0 <= y <= h - 1:
                pts.append((x, y))
    if abs(a) > 1e-12:
        for y in (0, h - 1):
            x = -(b * y + c) / a
            if 0 <= x <= w - 1:
                pts.append((x, y))
    if len(pts) < 2:
        return None
    return tuple(map(int, np.round(pts[0]))), tuple(map(int, np.round(pts[-1])))


def draw_epipolar(im1, im2, F, x1, x2, n=15):
    """n inlier matches; in each image the point and its epipolar line from the other."""
    a, b = im1.copy(), im2.copy()
    h, w = a.shape[:2]
    idx = RNG.choice(len(x1), min(n, len(x1)), replace=False)
    for k, i in enumerate(idx):
        col = tuple(int(c) for c in cv2.cvtColor(np.uint8([[[int(180 * k / n), 255, 230]]]), cv2.COLOR_HSV2BGR)[0, 0])
        for img, l, p in ((b, F @ np.append(x1[i], 1), x2[i]), (a, F.T @ np.append(x2[i], 1), x1[i])):
            seg = line_endpoints(l, w, h)
            if seg is not None:
                cv2.line(img, seg[0], seg[1], col, 1, cv2.LINE_AA)
            cv2.circle(img, tuple(np.round(p).astype(int)), 5, col, 2)
    return side_by_side(a, b)


def write_ply(path, X, colours):
    with open(path, "w") as f:
        f.write("ply\nformat ascii 1.0\n")
        f.write(f"element vertex {len(X)}\nproperty float x\nproperty float y\nproperty float z\n")
        f.write("property uchar red\nproperty uchar green\nproperty uchar blue\nend_header\n")
        for p, c in zip(X, colours):
            f.write(f"{p[0]:.5f} {p[1]:.5f} {p[2]:.5f} {int(c[2])} {int(c[1])} {int(c[0])}\n")


def plot_cloud(path, X, colours, cams, title):
    """Top view (x against z) and side view (z against -y) of the points and the
    camera centres with their optical axes."""
    lo, hi = np.percentile(X, 2, axis=0), np.percentile(X, 98, axis=0)
    keep = np.all((X >= lo - 0.2 * (hi - lo)) & (X <= hi + 0.2 * (hi - lo)), axis=1)
    X, colours = X[keep], colours[keep]
    rgb = colours[:, ::-1] / 255.0
    fig, ax = plt.subplots(1, 2, figsize=(12, 5.5))
    for a, (i, j, si, sj, lab) in zip(ax, ((0, 2, 1, 1, ("x", "z")), (2, 1, 1, -1, ("z", "-y")))):
        a.scatter(si * X[:, i], sj * X[:, j], s=1.5, c=rgb)
        span = np.ptp(np.r_[X[:, i], [c[0][i] for c in cams]]) * 0.08
        for k, (C, R) in enumerate(cams):
            d = R[2]
            a.plot(si * C[i], sj * C[j], "k^", ms=8)
            a.annotate(f"C{k + 1}", (si * C[i], sj * C[j]), textcoords="offset points", xytext=(5, 5))
            a.plot([si * C[i], si * (C[i] + span * d[i])], [sj * C[j], sj * (C[j] + span * d[j])], "k-")
        a.set_xlabel(lab[0])
        a.set_ylabel(lab[1])
        a.set_aspect("equal", adjustable="datalim")
        a.grid(alpha=0.3)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def reprojection_error(K, R, t, X, x):
    """Pixel distance between the projection of X and the observation x; inf for
    points behind the camera."""
    err = np.linalg.norm(project(K, R, t, X) - x, axis=1)
    err[depth_in(R, t, X) <= 0] = np.inf
    return err


def ray_angle_deg(C1, C2, X):
    a = X - C1
    b = X - C2
    cos = (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1))
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def load_gt(path):
    """R, t of a ground-truth 3x4 camera matrix P = K[R|t] (RQ decomposition)."""
    P = np.loadtxt(path)
    Q, Rr = np.linalg.qr(np.flipud(P[:, :3]).T)
    K = np.flipud(np.fliplr(Rr.T))
    R = np.flipud(Q.T)
    D = np.diag(np.sign(np.diag(K)))
    K, R = K @ D, D @ R
    t = np.linalg.solve(K, P[:, 3])
    if np.linalg.det(R) < 0:
        R, t = -R, -t
    return R, t


def gt_errors(gt, i, j, R, t):
    """Error of the recovered relative pose (R, t) of view j with respect to view
    i, against the ground truth: rotation angle, and angle of the translation."""
    (Ri, ti), (Rj, tj) = gt[i], gt[j]
    Rg = Rj @ Ri.T
    tg = tj - Rg @ ti
    return rotation_angle_deg(R.T @ Rg), angle_between_deg(t, tg), np.linalg.norm(tg)


def run_sfm(img1, img2, K, dist=None, gt=None, out="a4_out", max_side=1000, tol=1.0, reproj_tol=4.0,
            min_angle=1.0, show=True):
    """Given, do not change. Runs parts (b)-(d) on the photographs img1 and img2, prints every number the
    report asks for, writes the figures and report.txt to `out`, and shows the figures.
    K is one K.txt for both images (your phone) or a list with one K.txt per image (the sample data); a
    '# width W' header in the file is used to rescale it. dist is an optional dist.txt from the
    calibration. gt is an optional pair of true 3x4 camera matrices, used only to print the pose errors."""
    global RNG
    RNG = np.random.default_rng(0)          # same seed as the sample outputs, so the numbers are reproducible
    os.makedirs(out, exist_ok=True)
    lines = []

    def say(s=""):
        print(s)
        lines.append(s)

    t0 = time.time()
    images = [img1, img2]
    n_im = len(images)
    K_files = [K] if isinstance(K, str) else list(K)
    if len(K_files) not in (1, n_im):
        raise ValueError("give one K for both images, or one per image")
    ims, _ = load_images(images, max_side)
    h, w = ims[0].shape[:2]
    if any(im.shape[:2] != (h, w) for im in ims):
        raise ValueError("the two images must have the same size and orientation")
    Ks = [load_K(p, w, h)[0] for p in K_files]
    Ks = Ks * n_im if len(Ks) == 1 else Ks
    Kinv = [np.linalg.inv(Ki) for Ki in Ks]
    dist = np.loadtxt(dist).ravel() if dist else None
    gt = [load_gt(p) for p in gt] if gt else None
    say(f"{n_im} images at {w}x{h}")
    say("K (image 1) =\n" + np.array2string(Ks[0], precision=2, suppress_small=True))

    s = synthetic_tests()
    say(f"\nsynthetic checks (exact data): eight-point |F - F_true| = {s['F8']:.1e} on all points, "
        f"{s['F8min']:.1e} on a sample of eight; chirality counts {s['counts']}, "
        f"|R - R_true| = {s['R']:.1e}, |t - t_true| = {s['t']:.1e}; triangulation |X - X_true| = {s['X']:.1e}")

    feats = [sift_features(im) for im in ims]
    for i, (kp, _) in enumerate(feats):
        say(f"  image {i + 1}: {len(kp)} SIFT keypoints")
    m12 = match_pair(feats[0][1], feats[1][1])
    i1 = np.array([a for a, _ in m12])
    i2 = np.array([b for _, b in m12])
    x1 = keypoint_xy(feats[0][0], i1, Ks[0], dist)
    x2 = keypoint_xy(feats[1][0], i2, Ks[1], dist)
    say(f"  pair 1-2: {len(m12)} matches after the ratio and mutual tests")

    # (b) --------------------------------------------------------------
    F, inl, it = ransac_F(x1, x2, tol=tol)
    d = sampson_distance(F, x1, x2)
    say(f"\n(b) RANSAC on F: {inl.sum()}/{len(x1)} inliers ({100 * inl.mean():.1f}%) after {it} iterations; "
        f"RMS Sampson distance on inliers {np.sqrt((d[inl] ** 2).mean()):.3f} px")
    say(f"(b) singular values of F (||F|| = 1): {np.array2string(np.linalg.svd(F, compute_uv=False), precision=4)}")
    e, ep = epipoles(F)
    fmt = lambda v: f"({v[0] / v[2]:.0f}, {v[1] / v[2]:.0f})" if abs(v[2]) > 1e-9 * np.abs(v).max() else "at infinity"
    say(f"(b) epipoles: e = {fmt(e)} in image 1, e' = {fmt(ep)} in image 2 (images are {w}x{h})")
    cv2.imwrite(os.path.join(out, "b_inliers.jpg"), draw_inliers(ims[0], ims[1], x1, x2, inl))
    cv2.imwrite(os.path.join(out, "b_epipolar_lines.jpg"), draw_epipolar(ims[0], ims[1], F, x1[inl], x2[inl]))

    for name, flag in (("normalised", True), ("un-normalised", False)):
        Fx = eight_point(x1[inl], x2[inl], normalise=flag)
        sv = eight_point_singular_values(x1[inl], x2[inl], flag)
        dx = sampson_distance(Fx, x1[inl], x2[inl])
        say(f"(b) {name:13s} eight-point on the inliers: RMS Sampson {np.sqrt((dx ** 2).mean()):.3f} px, "
            f"sigma1/sigma8 of A = {sv[0] / sv[7]:.2e}, sigma8/sigma9 = {sv[7] / max(sv[8], 1e-300):.2e}")
    for t in (0.5, 1.0, 3.0):
        _, inl_t, _ = ransac_F(x1, x2, tol=t)
        say(f"(b) tolerance {t:3.1f} px: {inl_t.sum()} inliers")

    F_cv, mask_cv = cv2.findFundamentalMat(x1, x2, cv2.FM_RANSAC, tol, 0.999)
    mask_cv = mask_cv.ravel().astype(bool)
    d_cv = sampson_distance(F_cv, x1, x2)
    agree = (inl & mask_cv).sum() / max((inl | mask_cv).sum(), 1)
    say(f"(b) cv2.findFundamentalMat: {mask_cv.sum()} inliers, RMS Sampson on OUR inliers {np.sqrt((d_cv[inl] ** 2).mean()):.3f} px, "
        f"inlier sets agree on {100 * agree:.1f}% (intersection over union)")
    _, mask_h = cv2.findHomography(x1[inl], x2[inl], cv2.RANSAC, 3.0)
    say(f"(b) a single homography explains {100 * mask_h.mean():.1f}% of the F inliers at 3 px")

    # (c) --------------------------------------------------------------
    E = essential_from_F(F, Ks[0], Ks[1])
    x1n = from_h(to_h(x1[inl]) @ Kinv[0].T)
    x2n = from_h(to_h(x2[inl]) @ Kinv[1].T)
    R2, t2, counts = choose_pose(E, x1n, x2n)
    sv = np.linalg.svd(Ks[1].T @ F @ Ks[0], compute_uv=False)
    FE = Kinv[1].T @ skew(t2) @ R2 @ Kinv[0]
    dE = sampson_distance(FE, x1[inl], x2[inl])
    say(f"\n(c) singular values of K2^T F K1 before the correction, divided by the largest: {np.array2string(sv / sv[0], precision=4)}")
    say(f"(c) RMS Sampson distance of the F implied by the corrected E, on the F inliers: {np.sqrt((dE ** 2).mean()):.3f} px")
    say(f"(c) points in front of both cameras for the four candidates (Ra,+u3), (Ra,-u3), (Rb,+u3), (Rb,-u3): {counts}")
    say(f"(c) chosen pose: rotation of {rotation_angle_deg(R2):.2f} deg, t = {np.array2string(t2, precision=4)}")
    x1nn = from_h(to_h(x1[inl]) @ Kinv[0].T)
    x2nn = from_h(to_h(x2[inl]) @ Kinv[1].T)
    n_cv, R_cv, t_cv, _ = cv2.recoverPose(E, x1nn, x2nn, np.eye(3))
    say(f"(c) cv2.recoverPose on the same E: {n_cv} points in front; rotation differs from ours by "
        f"{rotation_angle_deg(R_cv.T @ R2):.3f} deg, translation direction by {angle_between_deg(t_cv.ravel(), t2):.3f} deg")
    if gt:
        eR, et, _ = gt_errors(gt, 0, 1, R2, t2)
        say(f"(c) against the ground truth: rotation error {eR:.3f} deg, translation direction error {et:.3f} deg")

    # (d) --------------------------------------------------------------
    R1, t1 = np.eye(3), np.zeros(3)
    X = triangulate(np.hstack([R1, t1[:, None]]), np.hstack([R2, t2[:, None]]), x1n, x2n)
    C1, C2 = centre(R1, t1), centre(R2, t2)
    e1 = reprojection_error(Ks[0], R1, t1, X, x1[inl])
    e2 = reprojection_error(Ks[1], R2, t2, X, x2[inl])
    ang = ray_angle_deg(C1, C2, X)
    front = (depth_in(R1, t1, X) > 0) & (depth_in(R2, t2, X) > 0)
    good = front & (ang > min_angle) & (e1 < reproj_tol) & (e2 < reproj_tol)
    say(f"\n(d) triangulated {len(X)} points: {front.sum()} in front of both, {(ang > min_angle).sum()} with ray angle > "
        f"{min_angle} deg, {good.sum()} kept after all three tests")
    say(f"(d) RMS reprojection error of the kept points: view 1 {np.sqrt((e1[good] ** 2).mean()):.3f} px, "
        f"view 2 {np.sqrt((e2[good] ** 2).mean()):.3f} px; ray angle median {np.median(ang[good]):.2f} deg, "
        f"10th percentile {np.percentile(ang[good], 10):.2f} deg")
    say(f"(d) median depth {np.median(depth_in(R1, t1, X[good])):.2f} baseline units (baseline ||C2 - C1|| = 1)")
    kp1 = feats[0][0]
    col1 = np.array([ims[0][int(round(kp1[i].pt[1])), int(round(kp1[i].pt[0]))] for i in i1[inl]])
    write_ply(os.path.join(out, "d_two_view.ply"), X[good], col1[good])
    plot_cloud(os.path.join(out, "d_two_view.png"), X[good], col1[good], [(C1, R1), (C2, R2)],
               f"two-view reconstruction, {good.sum()} points (unit = baseline)")

    say(f"done in {time.time() - t0:.1f} s, outputs in {out}/")
    with open(os.path.join(out, "report.txt"), "w") as f:
        f.write("\n".join(lines) + "\n")
    if show:
        show_image(os.path.join(out, "b_inliers.jpg"), "matches: RANSAC inliers green, outliers red")
        show_image(os.path.join(out, "b_epipolar_lines.jpg"), "15 inliers and their epipolar lines")
        show_image(os.path.join(out, "d_two_view.png"), "the point cloud, seen from above and from the side")
    return dict(F=F, inliers=inl, E=E, R=R2, t=t2, counts=counts, X=X[good])

### Run on the sample data

Run the pipeline on Wadham views 1 and 2, and compare your numbers with the reference `report.txt` printed below it.

In [ ]:
wadham = f"{DATA}/wadham"
res_wadham = run_sfm(f"{wadham}/001.jpg", f"{wadham}/002.jpg", K=[f"{wadham}/K1.txt", f"{wadham}/K2.txt"],
                     gt=[f"{wadham}/001.P", f"{wadham}/002.P"], out="out_wadham_12")

In [ ]:
print(open(f"{DATA}/sample_output/wadham_12/report.txt").read())
show_image(f"{DATA}/sample_output/wadham_12/d_two_view.png", "reference point cloud")

### Run on your own photographs

Upload your two scene photos with the cell below (or drag them into a folder `my_scene/` with the file browser on the left). The files are used in alphabetical order, so name them in the order you took them, e.g. `scene1.jpg`, `scene2.jpg`. The next cell uses the $K$ and distortion of your phone from part (a). The point cloud is also saved as `a4_out/d_two_view.ply`; you can download it and open it in MeshLab if you like.

In [ ]:
my_scene = upload_to("my_scene")

In [ ]:
assert len(my_scene) == 2, "upload exactly two scene photos with the cell above"
res_mine = run_sfm(my_scene[0], my_scene[1], K="a4_calib/K.txt", dist="a4_calib/dist.txt", out="a4_out")

### Report for (b) - (d)

Answer from the run on **your own photographs**. The printed numbers and the figures above are part of the report; refer to them.

**(b)**
1. Give the percentage of inliers and the RMS Sampson distance. Refer to the two figures (the inliers, and the epipolar lines).
2. Give the two epipoles. Where are they? Explain why, using how you moved the camera.
3. Compare the RMS Sampson distance with and without normalisation. Which is better?

**(c)** Give the number of points in front of both cameras for each of the four candidate poses, and the chosen rotation angle and translation $\mathbf t$. Does the direction of $\mathbf t$ match the way you moved?

**(d)** Give the number of points kept and the RMS reprojection error, and refer to the point-cloud figure. In what units are the 3D coordinates, and why? (See Question 5(c).)

**Your answer (b).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**Your answer (c).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

**Your answer (d).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

### (e) Experiments on the sample data  [6 marks]

You do not need new photographs for this part.

1. **A flat scene.** Run the code on the checkerboard pair `left01`, `left02` (first cell below). What fraction of the matches does a single homography explain? Do the four chirality counts give one clear winner? Explain using Question 4(c).
2. **A wider baseline.** Compare the Wadham pair (1,2), which you ran above, with the pair (1,3) (second cell below). Which pair has more matches? Which has larger ray angles (look at the median ray angle)? Why?

In [ ]:
# (e).1 a flat scene: two views of the sample checkerboard, with the K and distortion of the sample board
# (the same files that your run of part (a) on the sample board wrote to out_calib/).
calib = f"{DATA}/sample_output/calib"
res_planar = run_sfm(f"{DATA}/checkerboard/left01.jpg", f"{DATA}/checkerboard/left02.jpg",
                     K=f"{calib}/K.txt", dist=f"{calib}/dist.txt", out="out_planar")

In [ ]:
# (e).2 the wider pair: Wadham views 1 and 3
res_wadham13 = run_sfm(f"{wadham}/001.jpg", f"{wadham}/003.jpg", K=[f"{wadham}/K1.txt", f"{wadham}/K3.txt"],
                       gt=[f"{wadham}/001.P", f"{wadham}/003.P"], out="out_wadham_13")

**Your answer (e).**

*Type your answer here. Show the intermediate steps; LaTeX such as `$\lambda_1$` renders.*

---
*End of Assignment 4.*